# 07 · Latent Diffusion

> **Source notes:** `LatentDiffusion.md`

Diffusing in pixel space is expensive. Compress 8× first → diffuse → decode.

This notebook:
- Builds a **mini VAE** (encoder + decoder) trained on MNIST and Fashion-MNIST
- Visualises the 8-dim latent space (2D projection)
- Verifies the latent variance before and after the 0.18215 rescaling trick
- Runs a **drift-free latent DDPM** on the compressed space
- **PixelSmith v4** bonus: loads SDXL-Turbo via `diffusers` (CPU, ~2 min)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Mini VAE on MNIST

Compress 28×28=784 pixels → 8-dim latent. Matching SD's 8× spatial compression ratio in spirit.

In [ ]:
LATENT_DIM = 8

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        # Encoder: 784 → 256 → μ, logσ² (each latent_dim)
        self.encoder_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        # Decoder: latent_dim → 256 → 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 784),        nn.Tanh(),
        )
    
    def encode(self, x):
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterise(self, mu, logvar):
        std = (0.5 * logvar).exp()
        return mu + std * torch.randn_like(std)
    
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(recon, x, reduction="sum") / x.shape[0]
    kl_loss    = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1).mean()
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

# Train
dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)
vae    = VAE()
opt    = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(20):
    for x, _ in loader:
        recon, mu, logvar = vae(x)
        loss, _, _ = vae_loss(recon, x, mu, logvar, beta=0.1)
        opt.zero_grad(); loss.backward(); opt.step()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20  loss={loss.item():.2f}")
print("VAE training done.")

## 2 · Latent Space Visualisation

In [ ]:
from torchvision.datasets import MNIST
test_set = MNIST("/tmp/mnist", train=False, download=True,
                 transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
test_loader = DataLoader(test_set, batch_size=2048, shuffle=False)
x_test, y_test = next(iter(test_loader))

vae.eval()
with torch.no_grad():
    mu, logvar = vae.encode(x_test)
    z_test = vae.reparameterise(mu, logvar).numpy()

# PCA to 2D
mean  = z_test.mean(0, keepdims=True)
z_c   = z_test - mean
U, S, Vt = np.linalg.svd(z_c, full_matrices=False)
z_2d  = z_c @ Vt[:2].T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
scatter = ax1.scatter(z_2d[:, 0], z_2d[:, 1], c=y_test.numpy(), cmap="tab10", s=2, alpha=0.6)
plt.colorbar(scatter, ax=ax1, ticks=range(10))
ax1.set_title("VAE latent space (PCA 2D)")
ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2")

# Latent variance before and after scaling
var_before = z_test.var(0)
SCALE = 0.18215
z_scaled = z_test * SCALE
var_after  = z_scaled.var(0)
ax2.bar(range(LATENT_DIM), var_before, label="Before scaling", alpha=0.7)
ax2.bar(range(LATENT_DIM), var_after,  label=f"After ×{SCALE:.5f}", alpha=0.7)
ax2.axhline(1.0, color="red", linestyle="--", label="Target var=1")
ax2.set_xlabel("Latent dim"); ax2.set_ylabel("Variance")
ax2.set_title("Latent variance (SD scaling trick)"); ax2.legend()

plt.tight_layout(); plt.show()
print(f"Mean latent var before scaling: {var_before.mean():.3f}")
print(f"Mean latent var after  scaling: {var_after.mean():.4f}  (closer to 1.0)")

## 3 · Latent Diffusion: Train DDPM in Latent Space

In [ ]:
# Noise schedule
T_STEPS   = 500
betas     = torch.linspace(1e-4, 0.02, T_STEPS)
alphas    = 1 - betas
alpha_bar = torch.cumprod(alphas, 0)

# Small MLP denoiser (latent is only 8-dim, so no U-Net needed)
class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, time_dim=32):
        super().__init__()
        self.time_embed = nn.Embedding(T_STEPS, time_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_dim, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, latent_dim)
        )
    def forward(self, z, t):
        t_emb = self.time_embed(t)
        return self.net(torch.cat([z, t_emb], dim=1))

denoiser = LatentDenoiser()
d_optim  = torch.optim.Adam(denoiser.parameters(), lr=1e-3)

# Precompute all latent codes
vae.eval()
with torch.no_grad():
    all_x, all_y = [], []
    for xb, yb in DataLoader(dataset, batch_size=512, shuffle=False):
        mu, _ = vae.encode(xb)
        all_x.append(mu)
        all_y.append(yb)
Z_all = torch.cat(all_x, 0)   # (60000, 8)
print(f"Latent dataset shape: {Z_all.shape}  variance: {Z_all.var().item():.3f}")

from torch.utils.data import TensorDataset
lat_loader = DataLoader(TensorDataset(Z_all), batch_size=512, shuffle=True)

denoiser.train()
for epoch in range(30):
    for (z0,) in lat_loader:
        t   = torch.randint(0, T_STEPS, (z0.shape[0],))
        eps = torch.randn_like(z0)
        zt  = alpha_bar[t].sqrt().unsqueeze(1)*z0 + (1-alpha_bar[t]).sqrt().unsqueeze(1)*eps
        loss = F.mse_loss(denoiser(zt, t), eps)
        d_optim.zero_grad(); loss.backward(); d_optim.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/30  loss={loss.item():.4f}")
print("Latent denoiser trained.")

## 4 · Sample from Latent Diffusion → Decode to Pixels

In [ ]:
@torch.no_grad()
def latent_ddpm_sample(denoiser, n=16, n_steps=200):
    denoiser.eval()
    z = torch.randn(n, LATENT_DIM)
    # Use only a sub-sequence (DDIM-style) for speed
    ts = torch.linspace(T_STEPS-1, 0, n_steps).long()
    for i in range(len(ts)-1):
        t_cur  = ts[i]
        t_next = ts[i+1]
        tb     = torch.full((n,), t_cur, dtype=torch.long)
        eps    = denoiser(z, tb)
        ab_c   = alpha_bar[t_cur]
        ab_n   = alpha_bar[t_next]
        z0_p   = (z - (1-ab_c).sqrt()*eps) / ab_c.sqrt()
        z      = ab_n.sqrt()*z0_p + (1-ab_n).sqrt()*eps
    return z

z_samp = latent_ddpm_sample(denoiser, n=16)
vae.eval()
with torch.no_grad():
    imgs = vae.decode(z_samp).clamp(-1, 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
plt.suptitle("Latent DDPM → VAE decode → pixels (unconditional)")
plt.tight_layout(); plt.show()
print("Pipeline: sample z in latent → decode to image. No pixel-space noise!")

## 5 · PixelSmith v4 — SDXL-Turbo via `diffusers` (optional, CPU)

This cell downloads SDXL-Turbo (~7 GB) and runs 4-step generation on CPU. Skip if disk-constrained; it takes ~3 minutes on a modern CPU.

> Uncomment and run when ready.

In [ ]:
# # Uncomment to run — downloads SDXL-Turbo ~7 GB on first run
#
# subprocess.run([sys.executable, "-m", "pip", "install",
#                 "diffusers", "transformers", "accelerate", "safetensors", "-q"], check=True)
#
# from diffusers import AutoPipelineForText2Image
# import torch, time
#
# pipe = AutoPipelineForText2Image.from_pretrained(
#     "stabilityai/sdxl-turbo",
#     torch_dtype=torch.float32,   # CPU requires float32
# )
# pipe = pipe.to("cpu")
#
# prompt = "a photorealistic hero shot of PixelSmith studio, warm lighting, 4k"
# t0  = time.time()
# img = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
# print(f"Generated in {time.time()-t0:.1f}s")
#
# import matplotlib.pyplot as plt
# plt.figure(figsize=(6,6))
# plt.imshow(img)
# plt.axis("off")
# plt.title(f"PixelSmith v4: SDXL-Turbo 4-step\n{prompt[:60]}...")
# plt.show()

print("SDXL-Turbo cell is commented out. Uncomment to run (needs ~7 GB download).")

## 6 · Summary

```
Latent Diffusion vs. Pixel Diffusion

  Pixel-space DDPM (our MNIST model):
    Train + inference on 28×28×1 = 784 dims

  Latent-space DDPM (SD 1.x):
    1. VAE encode: 512×512×3 → 64×64×4 = 16384 dims  (48× cheaper)
    2. Diffuse in latent space: cross-attention with text
    3. VAE decode: 64×64×4 → 512×512×3

Key properties:
  - VAE trains separately (frozen during diffusion training)
  - Latents must be rescaled to unit variance (×0.18215 in SD)
  - Text conditioning enters via cross-attention, not embedding addition
  - Any DDIM/DPM-Solver scheduler applies unchanged to the latent denoising
```

**Next:** [TextToImage.md](../TextToImage/TextToImage.md) — prompt engineering, img2img, inpainting, and ControlNet.